# 03 · Future scenarios: a system-dynamics model of Myanmar

Where might Myanmar go from here? This notebook walks through Amber's future layer. A small stock-and-flow model is calibrated on 2011–2024 and run forward to 2035 under four scenarios. Everything here is a **scenario, not a forecast**: a way to see how the model's assumptions play out, not a prediction.

The logic lives in `src/amber/modeling/system_dynamics.py`, and `make sd` produces the same tables and charts.

**Prerequisites:** `make panel`, plus `make sc` for the phase 3 consistency check.

In [ ]:
%matplotlib inline
import logging
from dataclasses import asdict

import pandas as pd

from amber import config, dynamics, figures
from amber.config import Scenario
from amber.modeling import system_dynamics as sd

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.precision", 3)

panel, sc_paths, sc_metrics = dynamics.load_inputs()
future = sd.run(panel, sc_paths=sc_paths, sc_metrics=sc_metrics)
future.credible

## 1 · The model

There are four stocks: physical capital **K**, human capital **H**, connectivity **I** (internet users) and institutional stability **S**. Stability is set by the scenario. Output is

$$Y = A_0 (1+g)^{t}\,(1 - c)\,S^{\gamma}\,(1 + \kappa I/I_{max})\,K^{\alpha}H^{\beta}$$

The **feedback loop:** connectivity raises productivity, which raises output, which raises investment in capital and speeds the spread of connectivity. Stability multiplies productivity, so a more stable country gets more out of every part of the loop. The levers are multipliers on calibrated behaviour, where 1.0 means reform-era behaviour:

In [ ]:
dynamics.scenarios_table().pivot(index="scenario", columns="lever", values="value")

## 2 · Calibration: what's fitted and what isn't

Thirteen parameters are fitted by bounded least squares against Myanmar's 2011–2024 indicators, measured on the development index's own goalpost scale. Two are **deliberately fixed**. Refitting everything else with each one held at different values barely changed the fit. So the data can't identify either one, and each is set as a documented assumption:

- **κ**, how much connectivity raises productivity
- **the savings rate**

`at_bound` flags any fitted parameter that ends up at the edge of its allowed range.

In [ ]:
dynamics.calibration_table(future.calibration)

## 3 · Backtest

How well does the calibrated model reproduce history? This is an **in-sample** check: the effect of the coup can't be estimated without data from after 2021. If the error exceeds `SD_CREDIBLE_NRMSE`, every chart calls the scenarios illustrative dynamics rather than a calibrated projection.

In [ ]:
future.backtest.nrmse.to_frame().assign(n_obs=future.backtest.n_obs), future.backtest.overall

In [ ]:
figures.backtest_figure(future)

## 4 · Scenarios

Each scenario runs as a 200-member ensemble, with the calibrated parameters jittered around the fit. The bands span p10–p90. They show how sensitive the results are to the parameters, not a confidence interval.

The index here is computed on the **modeled indicators only**. Poverty and high-tech exports have no driver in the model, so they're excluded, and the level therefore differs from the published index. Compare the scenarios with each other, not with the phase 2 chart.

In [ ]:
figures.future_fan_figure(future)

**Checking against phase 3.** Over 2021–2024 the no-coup scenario should stay close to the synthetic control, wherever that control is credible. A large gap would be a red flag.

In [ ]:
pd.DataFrame([asdict(check) for check in future.sc_checks])

In [ ]:
figures.future_gdp_figure(future, sc_paths)

In [ ]:
figures.stocks_figure(future)

## 5 · Build your own scenario

This is what the phase 5 sliders will do. A scenario is just data: a stability path and some lever settings. The next cell sets out a slower, partial recovery with extra health spending:

In [ ]:
mine = Scenario(
    "slow_health_recovery",
    "Slow recovery, health push",
    stability=(*config.SD_HISTORICAL_STABILITY, (2024, 0.0), (2035, 0.3)),
    levers={"health_spend": 1.5},
)
path = sd.simulate(mine, future.calibration.params, future.calibration.initial)
path.loc[[2024, 2030, 2035], ["Y", "H", config.COMBINED_SERIES]]

## Reading the results

- The gap between no coup and actual continuation is large, and it holds across the range of κ the data can't distinguish.
- Reform push adds relatively little on top of no coup. By 2025 connectivity is already close to saturation in the model, so most of the extra comes from education.
- The model misses Myanmar's stagnation after 2021: it expects recovery growth, but the economy barely moved. Actual continuation is therefore likely **optimistic**.
- Myanmar's internet data stop in 2020, so nothing constrains how fast connectivity saturates after that.